# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a clinical and molecular dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata as a single object
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset covers multiple tables:
- Table 1: Clinical features and demographics
- Table 2: Hormone levels, imaging, and ultrasound
- Table 3: Genetic variants

We'll enumerate record sets using their `@id` and show their fields/columns by `@id`.

In [ ]:
# List available record sets and their IDs
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print("- Record Set Name:", getattr(rs, 'name', '(no name)'))
    print("  @id:", rs.id)
    print("  Fields:")
    for field in rs.fields:
        print("    - Field Name:", getattr(field, 'name', '(no name)'), "| @id:", field.id)
    print()

# For demonstration, print the first few records of the first record set
if len(record_sets) > 0:
    rs_id = record_sets[0].id
    print(f"Example records from record set {rs_id}:")
    for rec in dataset.records(record_set=rs_id):
        print(rec)
        break  # Only show one record for brevity

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather all record_set IDs
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records as a list of dicts
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded dataframe for Record Set @id: {record_set_id} ({df.shape[0]} rows, {df.shape[1]} columns)")

# Show available columns for first record set
first_rs = record_set_ids[0]
print("Columns for record set @id:", first_rs)
print(dataframes[first_rs].columns.tolist())

# Preview first 5 records
dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping. All field and column references use `@id`.

As an example, let's select `age` (likely labeled by a specific field `@id`) and filter records accordingly.

In [ ]:
# Find candidate numeric fields from the first record set
df = dataframes[first_rs]
numeric_field_id = None
for col in df.columns:
    # Try to find 'age' or another numeric column
    if 'age' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # Pick first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print("Using numeric field @id:", numeric_field_id)

# Filter: age > 10 (for demonstration; adjust threshold as needed)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field, e.g. 'infertility' or 'phenotype' if present
group_field_id = None
candidate_group_fields = ['infertility', 'phenotype', 'zygocity', 'pathogenicity']
for col in df.columns:
    for gf in candidate_group_fields:
        if gf in col.lower():
            group_field_id = col
            break
    if group_field_id:
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
Here, we'll plot the normalized values of the numeric field and, where available, compare by group field.

In [ ]:
# Plot histogram of normalized numeric field
plt.figure(figsize=(6,4))
filtered_df[f"{numeric_field_id}_normalized"].hist(bins=10)
plt.xlabel(f"{numeric_field_id}_normalized (field @id: {numeric_field_id})")
plt.ylabel("Count")
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.show()

# If grouped_df exists, show a bar plot
if group_field_id and 'grouped_df' in locals():
    plt.figure(figsize=(7,4))
    plt.bar(grouped_df[group_field_id].astype(str), grouped_df[numeric_field_id])
    plt.xticks(rotation=45)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and process a clinical genomics dataset using the `mlcroissant` library. All references to record sets, fields, and columns used the proper `@id`. You can extend this workflow for deeper analysis by selecting further record sets or fields, or for integrating additional Croissant-compliant datasets.